In [2]:
import serial
import time
from tools import Calculator
from errors import TLE

init_adp = '>02G9158'
init_dcm = '>01G6158'
secede = '>02Q5FD9'
query_adp = '>02g4959'    #查询运行状态
query_dcm = '>01dB819'

class controller:
    def __init__(self, serName, bps = 115200, timeout = 10):
        self.serName = serName
        self.bps = bps
        self.timeout = timeout
        self.ser = serial.Serial(self.serName, self.bps, timeout = 0)

    def wait_adp(self):
        self.ser.flushInput()
        self.ser.flushOutput()
        self.ser.write(query_adp.encode('utf-8'))
        time.sleep(0.1)
        stTime = time.time()
        while True:
            feedback = self.ser.read(20).decode('ascii', errors = 'ignore')

            self.ser.flushInput()
            self.ser.flushOutput()

            #print("get "+ feedback.strip())
            
            if feedback.strip() == '>02g01722E': #运行完成
                #print("done, exiting")
                return
                
            curTime = time.time()
            
            if curTime - stTime > self.timeout:
                raise TLE("mouth : ADP TLE")
            
            self.ser.write(query_adp.encode('utf-8'))
            time.sleep(0.1)

    def wait_dcm_position(self):
        self.ser.flushInput()
        self.ser.flushOutput()
        self.ser.write(query_dcm.encode('utf-8'))
        time.sleep(0.1)
        stTime = time.time()
        while True:
            #print("query sent")
            feedback = self.ser.read(40).decode('ascii', errors = 'ignore')

            #print("in wait_tj_dcm, feedback = " + feedback)
            
            self.ser.flushInput()
            self.ser.flushOutput()
            
            if feedback.strip() == '>01d0136DE':
                #print("done, exiting")
                return
                
            curTime = time.time()
            
            if curTime - stTime > self.timeout:
                raise TLE("mouth : DCM TLE")
            
            self.ser.write(query_dcm.encode('utf-8'))
            time.sleep(0.1)

    
    def init_adp(self):
        self.ser.write(init_adp.encode('utf-8'))
        time.sleep(0.1)
        self.wait_adp()

    def init_dcm(self):
        self.ser.write(init_dcm.encode('utf-8'))
        time.sleep(0.1)
        self.wait_dcm_position()

    def init(self):
        self.init_adp()
        self.init_dcm()

    def moveTo(self, idis):
        dis = int(idis / 100.0 * 150000)
        command = '>01D'
        list_dis = Calculator.intToList(dis, length = 4)
        for i in range(4):
            command += hex(list_dis[i])[2:].zfill(2)
        command += Calculator.crc(command.encode('utf-8'), 'str')
        self.ser.write(command.encode('utf-8'))
        time.sleep(0.1)
        self.wait_dcm_position()

    def secedeTip(self):
        self.ser.write(secede.encode('utf-8'))
        time.sleep(0.1)
        self.wait_adp()

    def suck(self, vol):
        command = '>02n'
        list_vol = Calculator.intToList(vol)
        command += str(hex(list_vol[0])[2:].zfill(2)) + str(hex(list_vol[1])[2:].zfill(2))
        command += Calculator.crc(command.encode('utf-8'), 'str')
        self.ser.write(command.encode('utf-8'))
        time.sleep(0.1)
        self.wait_adp()

    def spit(self, vol):
        command = '>02p'
        list_vol = Calculator.intToList(vol)
        command += str(hex(list_vol[0])[2:].zfill(2)) + str(hex(list_vol[1])[2:].zfill(2))
        command += Calculator.crc(command.encode('utf-8'), 'str')
        self.ser.write(command.encode('utf-8'))
        time.sleep(0.1)
        self.wait_adp()

    def close(self):
        self.ser.close()

if __name__ == '__main__':
    try:
        mouth = controller("COM9")
        mouth.init()
        #time.sleep(3)
        mouth.moveTo(50)
        #time.sleep(3)
        #mouth.suck(50)
        #time.sleep(3)
        #mouth.spit(0)
        #time.sleep(3)
        #mouth.secedeTip()
        mouth.moveTo(0)
        mouth.close()
    except Exception as e:
        mouth.close()
        raise